# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

## ⚙️ Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json, os

from _campaign_lib import *

svc = await init_services()
campaign_rounds = []

## 🎛️ Configuration

In [ ]:
campaign_config = {
    "queries_per_eval": 15,              # queries per optimization evaluation step
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "pipeline_overrides": {              # Override specific pipeline params (omitted = backend default):
        # "profiling_temperature": 0.3,
    },
    "optimization": {
        "n_variants": 5,
        "creativity": 0.7,
        "improvement_threshold": 0.01,
        "patience": 2,                   # rounds without improvement before auto-stop
        "max_rounds": 3,
    },
    "eval_llm": {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        "temperature": 0,
        "max_tokens": 4000,
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                   "descriptions to standardized database terms using entity profiling "
                   "and candidate ranking.",
        "grid_budget": 35,            # exact budget (0=full grid)
        "eval_queries_per_point": 6,  # queries per grid point (0=use all eval_data)
        "shared_queries": False,      # False=different random queries per point
        "seed": 42,
        "top_k": 5,
        "use_defaults": True,        # use DEFAULT_GRID_AXES library
    },
    "smart_search": {
        "n_diagnostic": 6,
        "max_rounds": 3,
        "stop_threshold": 0.0,
    },
}

## 📊 Data

In [ ]:
# Pipeline workflow config (synced from backend GET /pipeline)
pipeline_config = load_pipeline_config(svc["exp_data"])
print(json.dumps(pipeline_config, indent=2))

pipeline_params = build_pipeline_params(
    pipeline_config, overrides=campaign_config.get("pipeline_overrides"),
)
campaign_config["pipeline_params"] = pipeline_params

In [ ]:
#@title Backend status check
backend_status = await show_backend_status(svc["backend_client"])

In [ ]:
#@title Dataset summary
ds_summary = show_dataset_summary(svc["store"], svc["backend_id"])

In [ ]:
#@title Load datasets (Excel ground truth — alternative to trace-based eval_data)
# Set EXCEL_PATH to load from BOM-example.xlsx; leave empty to use traces
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"

if EXCEL_PATH:
    datasets = load_or_create_datasets(
        svc["store"], svc["backend_id"], EXCEL_PATH,
    )
    train_data = datasets["train"]
    svc["session_terms"] = build_all_session_terms(svc["store"], svc["backend_id"])
    print(f"\nTrain: {len(train_data)} queries | Session terms: {len(svc['session_terms'])}")
else:
    train_data = None
    print("No EXCEL_PATH set — using trace-based eval_data (next cell).")

In [ ]:
#@title 📡 Langfuse — cloud sync config
# Credentials: set LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY in .env
# Project name sets the Langfuse dataset name for this campaign's eval data.
LANGFUSE_PROJECT_NAME = "termnorm_ground_truth"
LANGFUSE_BACKFILL = True  # True = push all historical runs now
LANGFUSE_RESET = False     # True = clear push state first (re-push everything)

import api.services.obs.langfuse_push as _lfp
_lfp.DATASET_NAME = LANGFUSE_PROJECT_NAME

if LANGFUSE_BACKFILL:
    if LANGFUSE_RESET:
        from api.services.obs.langfuse_push import _state_path, _fresh_state, _save_state
        _save_state(svc["store"], svc["backend_id"], _fresh_state())
        print("Langfuse push state reset — will re-push all runs.")
    stats = push_langfuse(svc["store"], svc["backend_id"])

In [ ]:
# Load eval data: prefer Excel train set if available, else trace-based
if train_data:
    eval_data = train_data
    print(f"Using Excel train set: {len(eval_data)} queries")
else:
    eval_data = load_eval_dataset(svc["store"], svc["backend_id"], svc["experiment_id"])

cov_df = analyze_candidate_coverage(eval_data)
show_entity_profiles(eval_data)

In [ ]:
baseline = load_baseline_prompt(svc["exp_data"])
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
print(f"Evaluation data: {len(eval_data)} queries")

In [ ]:
#@title Evaluate baseline prompt
campaign_rounds, baseline_results = await run_baseline_eval(
    baseline, eval_data, campaign_config, svc,
)

## 🔍 Smart Search

In [ ]:
#@title Build diagnostic set (with resume)
variant_library = load_variant_library()
ss = campaign_config.get("smart_search", {})

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

_result = await resume_or_build_diagnostic(
    campaign_config, baseline, baseline_results,
    llm_client, llm_model,
    svc["store"], svc["backend_id"], eval_data,
    improvement_areas=campaign_config.get("improvement_areas", ""),
)
plan_id, search_baseline, diagnostic, diag_summary, cached_profiles = _result

In [ ]:
#@title Historical data audit
prompt_index = build_historical_index(svc["store"], svc["backend_id"])

# Try to synthesize sensitivity from grid data
if not cached_profiles:
    synth = synthesize_sensitivity(
        svc["store"], svc["backend_id"], prompt_index, diagnostic,
    )
    if synth:
        scan_df, axis_profiles = synth
        cached_profiles = axis_profiles
        print("Sensitivity derived from grid data — scan may be skippable.")

In [ ]:
#@title Data inventory
inventory = show_data_inventory(prompt_index, svc["store"], svc["backend_id"])

In [ ]:
#@title Coverage advisor
# Knobs: adjust these and re-run to see different strategies
min_queries = 6          # min queries per variant to count as "usable"
axis_requirements = None  # None = require all values; or e.g. {"persona": 2}

coverage = show_scan_coverage(
    search_baseline, variant_library, diagnostic,
    prompt_index,
    pipeline_params=campaign_config.get("pipeline_params"),
    min_queries=min_queries,
    axis_requirements=axis_requirements,
)

In [ ]:
#@title Sensitivity scan
if cached_profiles:
    print(f"[RESUME] Sensitivity scan already complete, "
          f"loaded {len(cached_profiles)} axis profiles")
    scan_df = None  # Not needed for adaptive search
    axis_profiles = cached_profiles
    display_axis_profiles(axis_profiles)
else:
    scan_df, axis_profiles = await sensitivity_scan(
        search_baseline, variant_library, diagnostic, svc.get("backend_client"),
        user_focus=campaign_config.get("improvement_areas", ""),
        store=svc["store"], backend_id=svc["backend_id"],
        pipeline_params=campaign_config.get("pipeline_params"),
        session_terms=svc.get("session_terms"),
        plan_id=plan_id,
        prompt_result_index=prompt_index,
    )

In [ ]:
#@title Select best from scan & seed campaign
best_ps, best_params = select_scan_winner_notebook(
    scan_df, axis_profiles, search_baseline, variant_library,
    pipeline_params=campaign_config.get("pipeline_params"),
    store=svc["store"], backend_id=svc["backend_id"], plan_id=plan_id,
)

if best_params:
    campaign_config["pipeline_params"] = best_params
    print(f"Updated pipeline_params: {best_params}")

campaign_rounds.append({
    "round": "search",
    "label": f"smart_search ({best_ps.changes_description or best_ps.id[:12]})",
    "prompt_state": best_ps,
    "accuracy": campaign_rounds[0]["accuracy"] if campaign_rounds else 0.0,
    "hits": campaign_rounds[0].get("hits", 0) if campaign_rounds else 0,
    "total": campaign_rounds[0].get("total", 0) if campaign_rounds else 0,
    "results": campaign_rounds[0].get("results", []) if campaign_rounds else [],
})
display_progress(campaign_rounds)

## 🗺️ Grid Search (Optional)

<details>
<summary>Skip if you used Smart Search above. Expand for brute-force grid sweep.</summary>

**What:** Systematic sweep of the prompt configuration space (Layer 1 fields) using a cartesian product of default axis variations. Maps the accuracy landscape before hill-climbing.

**When to use:** When you want exhaustive coverage of the grid, or when Smart Search results look unreliable and you want independent validation.

**What you get:** Ranked starting points, which dimensions matter most (marginal stats), interaction effects between fields (heatmaps), and LLM-analyzed insights.

**How to read results:**
- **Ranked table** — best combos at the top; use the winner as your campaign seed
- **Marginal stats** — which axis values have the highest mean accuracy across all combos
- **Pairwise heatmaps** — green = good interaction, red = bad; look for synergies and conflicts
- **LLM analysis** — automated pattern recognition across the grid results

</details>

In [ ]:
#@title Grid campaign overview (existing plans)
merge_plans = False  # Set True to combine results from multiple plans
grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
merged_grid_df = grid_overview.get("merged_grid_df")

In [ ]:
#@title Build or resume grid plan + load eval data
gs = campaign_config["grid_search"]
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

(
    grid_plan_id, grid_points, grid_state_lookup,
    grid_axes, layer1_fields, grid_baseline,
) = await resume_or_build_grid(
    campaign_config, baseline, llm_client, llm_model,
    svc["store"], svc["backend_id"],
    improvement_areas=campaign_config.get("improvement_areas", ""),
)

# Load full eval_data for later optimization rounds
eval_data = load_eval_dataset(
    svc["store"], svc["backend_id"], svc["experiment_id"],
)
if not eval_data:
    raise RuntimeError(
        "No evaluation data found. Generate data first "
        "(e.g. run termnorm_backend.ipynb or another data source)."
    )

print(f"Grid points: {len(grid_points)}")
print(f"Plan ID: {grid_plan_id}")

In [ ]:
#@title Run grid search
grid_df = await run_grid_search(
    grid_points, grid_state_lookup, eval_data,
    campaign_config["eval_llm"],
    plan_id=grid_plan_id,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_client=svc.get("backend_client"),
    session_terms=svc.get("session_terms"),
    pipeline_params=campaign_config.get("pipeline_params"),
    eval_queries_per_point=gs.get("eval_queries_per_point", 1),
    shared_queries=gs.get("shared_queries", False),
    grid_seed=gs.get("seed", 42),
)

In [ ]:
#@title Display grid results
_display_df = merged_grid_df if merged_grid_df is not None else grid_df
display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
#@title LLM analysis of grid results
_analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
grid_analysis = await analyze_grid_results(
    _analysis_df, grid_axes, llm_client, model=llm_model,
)

In [ ]:
#@title Select grid winner and seed campaign
grid_winner = select_and_seed_grid_winner(
    grid_df, merged_grid_df, grid_state_lookup,
    grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
)

## 🚀 Optimization

<details>
<summary>Details</summary>

Two modes:
- **Semi-automatic** (recommended): runs multiple rounds with patience-based auto-stop
- **Manual**: run one round at a time for full HITL control

Both modes subsample `eval_data` to `queries_per_eval` queries per step.

</details>

In [ ]:
#@title Run optimization (feedback cycle — M3 nodes)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
)

In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, svc,
)

## 💡 LLM Suggestions

<details>
<summary>Details</summary>

After each round, the LLM analyzes failures and suggests:
1. Failure pattern analysis
2. Parameter change suggestions
3. Prompt phrase fragments to adopt
4. Suggested next `campaign_config`

**Review the suggestions, edit the config cell (Section 2), then re-run Sections 5-6.**

</details>

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print(f"
--- SUGGESTED CONFIG (copy to Section 2) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

## 📋 Results

<details>
<summary>Details</summary>

Compare all rounds, track per-query flips, display the PromptState lineage chain, and save the winner.

</details>

In [ ]:
#@title Campaign comparison table
rows = []
for rd in campaign_rounds:
    rows.append({
        "round": rd["round"],
        "label": rd["label"][:40],
        "hit@1": rd["hits"],
        "total": rd["total"],
        "accuracy": f"{rd['accuracy']:.1%}",
        "prompt_id": rd["prompt_state"].id[:12],
    })

print(f"CAMPAIGN SUMMARY ({len(campaign_rounds)} rounds)")
print(f"{'='*70}")
display(pd.DataFrame(rows))

In [ ]:
#@title Per-query flip tracking (baseline vs final)
if len(campaign_rounds) >= 2:
    base_r = campaign_rounds[0]["results"]
    final_r = campaign_rounds[-1]["results"]

    flips = []
    for br, fr in zip(base_r, final_r):
        b_hit = br["hit"]
        f_hit = fr["hit"]
        if b_hit != f_hit:
            flips.append({
                "query": br["query"][:50],
                "flip": "MISS->HIT" if f_hit else "HIT->MISS",
                "base_pred": br["predicted"][:35],
                "final_pred": fr["predicted"][:35],
                "ground_truth": br["ground_truth"][:35],
            })

    gained = sum(1 for f in flips if f["flip"] == "MISS->HIT")
    lost = sum(1 for f in flips if f["flip"] == "HIT->MISS")

    print(f"FLIP TRACKING (baseline -> round {campaign_rounds[-1]['round']})")
    print(f"  Queries gained (MISS->HIT): {gained}")
    print(f"  Queries lost (HIT->MISS):   {lost}")
    print(f"  Net change:                 {gained - lost:+d}")
    print()
    if flips:
        display(pd.DataFrame(flips))
else:
    print("Need at least 2 rounds for flip tracking.")

In [ ]:
#@title PromptState lineage chain
print("LINEAGE CHAIN")
print("="*50)
for i, rd in enumerate(campaign_rounds):
    ps = rd["prompt_state"]
    parent = ps.parent_id[:12] if ps.parent_id else "root"
    arrow = "  " if i == 0 else "  -> "
    print(f"{arrow}[{ps.id[:12]}] Round {rd['round']}: {rd['label'][:40]} ({rd['accuracy']:.1%})")
    if ps.parent_id:
        print(f"       parent: {parent}  |  changes: {ps.changes_description or 'none'}")

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])

In [ ]:
#@title 📡 Sync evaluation history to Langfuse
# Backfill: pushes any eval runs not already synced to Langfuse cloud.
# Safe to re-run — already-pushed runs are skipped automatically.
import api.services.obs.langfuse_push as _lfp
_lfp.DATASET_NAME = LANGFUSE_PROJECT_NAME
stats = push_langfuse(svc["store"], svc["backend_id"])